# Pork rasher defect detection: Colab workflow

Thin notebook: all logic lives in the `porkrasher` package. Use a GPU runtime (Runtime > Change runtime type > T4 GPU).


In [ ]:
# ===== ASSESSOR / USER: SET THE DATASET ZIP PATH (the only line you must edit) =====
# The dataset is not in the repository. Use the Roboflow zip provided with the unit
# ("Pork Rasher Error-Packaging-.v4i.yolo26.zip") in ONE of two ways:
#
#   Option A (Google Drive): put the zip in your Drive, run the "drive.mount" cell below,
#            and set the full path, e.g.
#            ZIP_PATH = "/content/drive/MyDrive/<your folder>/Pork Rasher Error-Packaging-.v4i.yolo26.zip"
#
#   Option B (no Drive): drag the zip into the Files panel (folder icon, left side) and use
#            ZIP_PATH = "/content/Pork Rasher Error-Packaging-.v4i.yolo26.zip"
#            (you can then skip the drive.mount cell)

REPO_URL = "https://github.com/Nicholas-2305/MMA3001-Project-Nicholas-Caciolo-34988858.git"
ZIP_PATH = "/content/Pork Rasher Error-Packaging-.v4i.yolo26.zip"  # <- EDIT


In [ ]:
!git clone $REPO_URL repo
%cd repo
!pip install -q -e .

# Make the package importable in this session without restarting the runtime
import sys
sys.path.insert(0, "/content/repo/src")


In [ ]:
# Only needed for Option A (zip stored in Google Drive); skip for Option B
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from porkrasher.data import extract_dataset, write_data_yaml
root = extract_dataset(ZIP_PATH, '/content/dataset', overwrite=True)
data_yaml = write_data_yaml(root)
print(root, data_yaml)


In [ ]:
!python -m porkrasher.data --root /content/dataset


In [ ]:
from porkrasher.visualise import plot_samples
fig = plot_samples('/content/dataset/train/images', '/content/dataset/train/labels', n=4, seed=0)


## Training

YOLOv8n is the selected model. The YOLOv8s cell after it is an **optional comparison** (used in the report's model comparison); skip it to save about an hour. Later cells use YOLOv8s only if it was trained.


In [ ]:
from porkrasher.train import train
best_n = train('yolov8n.pt', data_yaml, epochs=100, name='yolov8n_100ep')
print(best_n)


In [ ]:
# OPTIONAL: comparison model (not the final model). Skip to save about an hour.
best_s = train('yolov8s.pt', data_yaml, epochs=100, name='yolov8s_100ep')
print(best_s)


## Validation split (use for model selection)

In [ ]:
from porkrasher.evaluate import evaluate, format_results_table, format_per_class_table
res_n = evaluate(best_n, data_yaml, split='val', out_json='results/yolov8n_val.json')
results = {'yolov8n': res_n}
if 'best_s' in globals():  # only if the YOLOv8s comparison model was trained
    results['yolov8s'] = evaluate(best_s, data_yaml, split='val', out_json='results/yolov8s_val.json')
print(format_results_table(results))
print(format_per_class_table(results, 'map50'))


## Test split

**Warning:** run this once, on the final chosen model only. Do not tune anything after seeing these numbers.

In [ ]:
res_test = evaluate(best_n, data_yaml, split='test', out_json='results/yolov8n_test.json')
print(format_results_table({'yolov8n': res_test}))


In [ ]:
from porkrasher.evaluate import benchmark_inference
from porkrasher.data import list_images
print(benchmark_inference(best_n, list_images('/content/dataset/test/images')))


Copy `best.pt` for both models to Drive so they survive the session.

In [ ]:
!cp {best_n} /content/drive/MyDrive/yolov8n_rerun_best.pt  # new name: do not overwrite the model the report was based on


## Tests and documentation

Run from the repository root. These produce the test report and HTML docs required for submission.

In [ ]:
!pip install -q pytest pdoc
!python -m pytest -v | tee results/test_report.txt


In [ ]:
!rm -f docs/*.html
!pdoc -o docs porkrasher
!zip -qr qa_outputs.zip docs results/test_report.txt  # download this from the Files panel
